In [3]:
import os, json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
os.environ["HF"] = os.getenv("HF")
LLM = ChatGroq(model=os.getenv("GROQ_MODEL"), api_key=os.getenv("GROQ_API_KEY"))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/pradiptabhuin/FastApi/transaction-assistant-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21463.30it/s]


In [10]:
# write the function to take the nesscery feature from the transaction.json
def format_transaction(record: dict, metadata: dict):
    metadata["id"] = record.get("id")
    metadata["status"] = record.get("status")
    metadata["type"] = record.get("type")
    metadata["amount"] = record.get("amount")
    metadata["name"] = record.get("name")
    return metadata


loader = JSONLoader(
    file_path="transactions.json",
    jq_schema=".[]",
    text_content=False,
    metadata_func=format_transaction,
)

docs = loader.load()

# rewrite the page content in human redable format
for doc in docs:
    data = json.loads(doc.page_content)
    direction = "Sent to" if data.get("type") == "debit" else "Recived from"
    doc.page_content=(
        f"Transaction ID: {data.get("id")} | "
        f"{direction} {data.get("name")}    ({data.get("upiId")}) | "
        f"Amount: {data.get("amount")} | "
        f"Status: {data.get("status")} | "
        f"Method: {data.get("paymentMethod")} | "
        f"Date: {data.get("date")} | "
        f"Note: {data.get("note")} | "
        f"Bank Ref: {data.get("bankRef")}"
    )